<a href="https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The purpose of this playbook is to turn the model output into practical, human-reviewed actions. The model is used as decision-support, not as an automatic diagnosis.

I rank participants using the model's predicted high-risk probability. Higher predicted probability receives a higher priority for human review.

Each recommendation also includes a reason code based on the participant's available features. The reason codes are intended to explain why a participant received attention in the queue, rather than claiming that a feature directly causes colorectal cancer.

The main action levels are:

* **High priority review:** Higher predicted risk; review the participant's available information first.
* **Standard review:** Lower predicted risk but still suitable for routine review.
* **No immediate action:** Lower predicted risk with no additional model-based action.

Example reason codes include higher age, higher BMI, family history of CRC, and lifestyle indicators. These are directional signals observed by the model and should be reviewed together rather than treated as individual diagnoses.


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/crc_dataset.csv")

df["Pre-existing Conditions"] = df["Pre-existing Conditions"].fillna("None")

encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

X = df.drop(["Participant_ID", "CRC_Risk"], axis=1)
y = df["CRC_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

risk_probability = rf.predict_proba(X_test)[:, 1]

queue = df.iloc[X_test.index].copy()
queue["Predicted_Risk_Probability"] = risk_probability

queue["Priority"] = pd.cut(
    queue["Predicted_Risk_Probability"],
    bins=[-0.01, 0.33, 0.66, 1.0],
    labels=["No Immediate Action", "Standard Review", "High Priority Review"]
)

queue = queue.sort_values(
    "Predicted_Risk_Probability",
    ascending=False
)

queue[[
    "Participant_ID",
    "Predicted_Risk_Probability",
    "Priority"
]].head(10)

,Participant_ID,Predicted_Risk_Probability,Priority
976,1977,0.750,High Priority Review
751,1752,0.730,High Priority Review
208,1209,0.725,High Priority Review
126,1127,0.710,High Priority Review
676,1677,0.695,High Priority Review
332,1333,0.695,High Priority Review
54,1055,0.645,Standard Review
413,1414,0.635,Standard Review
575,1576,0.605,Standard Review
412,1413,0.570,Standard Review


Reason-code interpretation

The reason codes are simple rule-based explanations added to the ranked queue. They are not causal explanations and should not be interpreted as proof that a particular factor caused the participant's predicted risk.

The purpose is to help a human reviewer understand which available characteristics contributed to the review priority. The reviewer should consider the complete available information before taking any action.

In [ ]:
def create_reason_codes(row):
    reasons = []

    if row["Age"] >= df["Age"].median():
        reasons.append("AGE_DIRECTIONAL_SIGNAL")

    if row["BMI"] >= df["BMI"].median():
        reasons.append("BMI_DIRECTIONAL_SIGNAL")

    if row["Family_History_CRC"] == "Yes":
        reasons.append("FAMILY_HISTORY_SIGNAL")

    if row["Lifestyle"] == "Smoker":
        reasons.append("LIFESTYLE_SIGNAL")

    if not reasons:
        reasons.append("NO_STRONG_RULE_BASED_SIGNAL")

    return ", ".join(reasons[:3])

queue["Reason_Codes"] = queue.apply(
    create_reason_codes,
    axis=1
)

queue = queue.sort_values(
    "Predicted_Risk_Probability",
    ascending=False
)

queue[[
    "Participant_ID",
    "Predicted_Risk_Probability",
    "Priority",
    "Reason_Codes"
]].head(10)

,Participant_ID,Predicted_Risk_Probability,Priority,Reason_Codes
976,1977,0.750,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
751,1752,0.730,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
208,1209,0.725,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
126,1127,0.710,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
676,1677,0.695,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
332,1333,0.695,High Priority Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
54,1055,0.645,Standard Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
413,1414,0.635,Standard Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
575,1576,0.605,Standard Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"
412,1413,0.570,Standard Review,"AGE_DIRECTIONAL_SIGNAL, BMI_DIRECTIONAL_SIGNAL"


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended for research and decision-support purposes. A human reviewer can use the ranked queue to identify which records should be reviewed first and to understand the directional signals associated with each model output.

The ranking can help organize review effort by placing higher predicted-risk records earlier in the queue.

### Limits

The model was developed and evaluated on the available colorectal cancer dataset. Its measured performance should not be assumed to represent performance on a different population or real clinical setting.

The Week-6 audit also identified limitations in the validation design. The dataset does not contain a true client identifier or time variable for stronger client-level or time-aware validation.

The model should therefore not be treated as a clinical diagnostic system. A high predicted probability does not mean that a participant has colorectal cancer, and a low predicted probability does not rule it out.

The output is directional decision-support and requires human review.


In [ ]:
print("Queue size:", len(queue))
print("\nPriority counts:")
print(queue["Priority"].value_counts())

print("\nRisk probability summary:")
print(queue["Predicted_Risk_Probability"].describe())

Queue size: 200

Priority counts:
Priority
No Immediate Action     166
Standard Review          28
High Priority Review      6
Name: count, dtype: int64

Risk probability summary:
count    200.000000
mean       0.150450
std        0.185903
min        0.000000
25%        0.020000
50%        0.055000
75%        0.230000
max        0.750000
Name: Predicted_Risk_Probability, dtype: float64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Before acting on any recommendation, a reviewer should:

1. Check the participant's available information.
2. Confirm that the data appears complete and reasonable.
3. Review the model probability together with the reason codes.
4. Check for missing or unusual values.
5. Avoid treating the prediction as a diagnosis.
6. Record the final human decision separately from the model recommendation.

### What should NOT be automated

The following actions should not be automated using this model:

* Diagnosing colorectal cancer.
* Automatically referring or rejecting a participant.
* Automatically starting or stopping medical treatment.
* Automatically communicating a medical diagnosis to a participant.
* Making decisions based only on one feature such as age or BMI.
* Treating model probability as a confirmed medical probability.
* Replacing a qualified professional's judgement.

The model output should only support prioritisation and review. The final decision remains with a human reviewer.


In [ ]:
no_go_actions = [
    "Automatic diagnosis",
    "Automatic treatment decision",
    "Automatic referral decision",
    "Automatic rejection",
    "Automatic patient notification of diagnosis"
]

for action in no_go_actions:
    print("NO-GO:", action)

NO-GO: Automatic diagnosis
NO-GO: Automatic treatment decision
NO-GO: Automatic referral decision
NO-GO: Automatic rejection
NO-GO: Automatic patient notification of diagnosis


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored because model behaviour can change when the input data changes.

The following conditions would trigger a review:

* The distribution of important features changes substantially.
* The proportion of high-risk predictions changes unexpectedly.
* Recall for the high-risk class decreases during later evaluation.
* False-negative cases increase.
* New data contains categories or values that were not present during training.
* Data collection or feature definitions change.
* New independent evaluation data shows a meaningful drop in F1-score or recall.

Retraining should not happen automatically after every small change. A retraining review should first confirm that the new data is reliable, relevant, and sufficiently representative.

A new model should be compared against the current model using the same evaluation process before it replaces the existing version.


In [ ]:
print("Monitoring checks")

print("\nCurrent high-priority percentage:")
high_priority_rate = (
    (queue["Priority"] == "High Priority Review").mean() * 100
)

print(round(high_priority_rate, 2), "%")

print("\nCurrent average predicted risk:")
print(round(queue["Predicted_Risk_Probability"].mean(), 4))

print("\nRetrain review triggers:")
print("- Significant feature distribution change")
print("- Increased false negatives")
print("- Lower recall or F1-score on new evaluation data")
print("- Changes in feature definitions")
print("- New data categories not seen during training")

Monitoring checks

Current high-priority percentage:
3.0 %

Current average predicted risk:
0.1504

Retrain review triggers:
- Significant feature distribution change
- Increased false negatives
- Lower recall or F1-score on new evaluation data
- Changes in feature definitions
- New data categories not seen during training


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported so that the recommendations section of the research paper can use the same outputs produced by this notebook.

The exported queue contains the participant identifier, measured model risk probability, priority level, and reason codes.

The queue is generated by the notebook rather than manually edited. This makes the paper's recommendations traceable to the analysis.

The exported file is intended for the research workflow and should not be treated as a production operational queue.


In [ ]:
output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

export_columns = [
    "Participant_ID",
    "Predicted_Risk_Probability",
    "Priority",
    "Reason_Codes"
]

queue_export = queue[export_columns].copy()

queue_path = os.path.join(
    output_dir,
    "w07_ranked_action_queue.csv"
)

queue_export.to_csv(
    queue_path,
    index=False
)

print("Export created:")
print(queue_path)

print("\nExport shape:")
print(queue_export.shape)

queue_export.head(10)

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.